# IronStore Enterprise AI Assistant — v3 (product chunks · routing · reranker)

<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

**What changed** (`v1` and `v2` notebooks stay untouched):

| | v1 | v2 | 🆕 v3 |
|---|---|---|---|
| Catalog chunks | fixed 450 tokens | fixed 450 tokens | **one chunk = one product** + metadata `sku`, `price`, `category`, `availability` |
| Search | dense | hybrid (dense + keywords) | hybrid **+ metadata filters** |
| Routing | — | α only | an LLM **router** decides: *policy / product / both*, exact **SKU**, **max price**, old catalog wanted? |
| SKU question | ❌ | ✅ via keywords | ✅ **direct metadata lookup** (exact, fastest) |
| Outdated 2021 catalog | searched with the rest | searched with the rest | **excluded**, used only as a **fallback** (flagged) |
| Ranking | vector score | hybrid score | **reranker** `bge-reranker-v2-m3` reads question + chunk together |
| Pinecone index | `ironstore-assistant` | `ironstore-hybrid` | `ironstore-v3` |

**Legend:** <span style="background:#f3e5f5;border-left:4px solid #7b1fa2;padding:0 6px">🆕 purple = new in v3</span> &nbsp;
<span style="background:#f3f3f3;border-left:4px solid #9e9e9e;padding:0 6px">♻️ grey = same as v2</span>

**Pipeline:** question → 🧭 router ∥ ✍️ 3 rewrites → 🔎 hybrid search with filters → 🧹 dedup → 🏅 rerank → 🤖 answer + sources
</div>


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Step 0 — Speed fix (this PC only) <span style="color:#757575;font-size:0.8em">(same as v2)</span>

Try IPv4 before IPv6 so each API call takes ~1 s instead of ~40 s.

</div>

In [1]:
import socket

if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 1 — Settings

Same as v2 plus the **reranker** model, the **old-catalog fallback** level, and a new index name.

</div>

In [2]:
import os
import re
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()

DOCS_DIR = Path(r"C:\Users\Windows\Desktop\Ironhack\04_llm\project-brief-enterprise-ai-assistant\documents\extracted\documents")
INDEX_NAME = "ironstore-v3"                     # 🆕 v3
EMBED_MODEL = "text-embedding-3-small"          # dense: meaning
EMBED_DIM = 1536
SPARSE_MODEL = "pinecone-sparse-english-v0"     # sparse: exact words
RERANK_MODEL = "bge-reranker-v2-m3"             # 🆕 v3: reranker hosted by Pinecone
FALLBACK_RERANK = 0.5                           # 🆕 v3: product match weaker than this → also try the 2021 catalog
TOP_N = 6                                       # 🆕 v3: chunks the answer model gets after reranking
ROUTER_MODEL = REWRITE_MODEL = "gpt-4o-mini"    # small, fast jobs
ANSWER_MODEL = "gpt-4.1-mini"                   # the answer

print("OpenAI key found:  ", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key found:", bool(os.getenv("PINECONE_API_KEY")))

OpenAI key found:   True
Pinecone key found: True


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Step 2 — Load the PDFs <span style="color:#757575;font-size:0.8em">(same as v2)</span>

One record per document with area, title, last reviewed, outdated flag and page positions.

</div>

In [3]:
from bisect import bisect_right
import pymupdf

def read_pdf(path: Path) -> dict:
    pages = [page.get_text() for page in pymupdf.open(path)]
    text = "".join(pages)
    head = text[:600]
    match = re.search(r"Last reviewed(?: in|:)?\s*([A-Z][a-z]+ \d{4})", head)
    last_reviewed = match.group(1) if match else "unknown"
    year = int(last_reviewed[-4:]) if match else 0
    if path.parent.name == "product_catalog":
        title = path.stem.replace("IronStore_", "").replace("_", " ")
    else:
        title = head.strip().splitlines()[0]
    page_starts, pos = [], 0
    for page in pages:
        page_starts.append(pos)
        pos += len(page)
    return {"path": path, "file": path.name, "area": path.parent.name,
            "doc_type": "catalog" if path.parent.name == "product_catalog" else "policy",
            "title": title, "last_reviewed": last_reviewed, "outdated": 0 < year < 2026,
            "text": text, "page_starts": page_starts}

docs = [read_pdf(p) for p in sorted(DOCS_DIR.rglob("*.pdf"))]
print(len(docs), "documents")

83 documents


<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 3 — Product-aware chunking

The 63 catalogs use **4 different layouts** (numbered lists, `CODE — Name`, tables, `Page 4 — SKU`…), but every product has a **`Category:`** line near its top. So we:
1. find every `Category:` line,
2. walk **up** over the `SKU:` / `Product title:` lines and the product **name** (and a `Page …` header just above) → that's where the product starts,
3. cut the catalog there → **one chunk per product** (a rare very long block is split again),
4. read `sku`, `name`, `price`, `category`, `availability` into **metadata** (for filters).

Policies keep the v2 token splitter — it worked well.

</div>

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

BULLET = r"(?:•\s*)?"
CATEGORY_LINE = re.compile(rf"^{BULLET}Category:")
FIELD_LINE = re.compile(rf"^{BULLET}(?:SKU|Product title|Product code|Item code|Brand)\b")
PAGE_LINE = re.compile(r"^Page \d+")
CODE = r"[A-Z]{2,}[A-Z0-9]*(?:-[A-Z0-9]+)+"

def product_starts(text):
    lines, offsets, pos = text.split("\n"), [], 0
    for line in lines:
        offsets.append(pos)
        pos += len(line) + 1
    starts = []
    for i, line in enumerate(lines):
        if not CATEGORY_LINE.match(line):
            continue
        j = i - 1
        while j >= 0 and FIELD_LINE.match(lines[j]):   # SKU / Product title …
            j -= 1
        if j >= 0 and not PAGE_LINE.match(lines[j]):   # the product name
            j -= 1
        if j >= 0 and PAGE_LINE.match(lines[j]):       # "Page 4 — …" header
            j -= 1
        starts.append(offsets[j + 1])
    return starts

def product_metadata(block):
    def field(pattern):
        match = re.search(pattern, block, re.M)
        return match.group(1).strip() if match else ""
    name = next(l for l in block.split("\n") if l.strip() and not PAGE_LINE.match(l))
    name = re.sub(rf"^(?:\d+\.\s+|{CODE}\s+[—–-]\s+)", "", name.strip())
    sku = field(rf"SKU:\s*({CODE})") or field(rf"^(?:Page[^\n]*\n)?({CODE})\s+[—–-]")
    price = field(r"Price:\s*€\s*([\d.,]+)").replace(",", "")
    meta = {"name": name, "category": field(rf"^{BULLET}Category:\s*(.+)$"),
            "availability": field(rf"^{BULLET}Availability:\s*(.+)$")}
    if sku:
        meta["sku"] = sku
    if price:
        meta["price"] = float(price)
    return meta

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=450, chunk_overlap=60)

def make_chunk(doc, n, start, body, extra=None):
    page = bisect_right(doc["page_starts"], start)
    header = (f"[{doc['title']} | {doc['area']} | page {page} | last reviewed {doc['last_reviewed']}"
              + (" | OUTDATED" if doc["outdated"] else "") + "]")
    return {"id": f"{doc['path'].stem}-{n:04d}", "text": header + "\n" + body.strip(),
            "source": doc["file"], "title": doc["title"], "area": doc["area"], "doc_type": doc["doc_type"],
            "page": page, "last_reviewed": doc["last_reviewed"], "outdated": doc["outdated"],
            "path": str(doc["path"]), **(extra or {})}

def split_text(doc, text, offset, first_n, extra=None):
    out, start = [], 0
    for piece in splitter.split_text(text):
        start = max(text.find(piece, start), start)
        out.append(make_chunk(doc, first_n + len(out), offset + start, piece, extra))
    return out

chunks = []
for doc in docs:
    text = doc["text"]
    if doc["doc_type"] == "policy":
        chunks += split_text(doc, text, 0, 0, {"kind": "policy"})
        continue
    starts = product_starts(text)
    doc_chunks = split_text(doc, text[:starts[0]], 0, 0, {"kind": "catalog intro"})   # intro before 1st product
    for a, b in zip(starts, starts[1:] + [len(text)]):
        block, meta = text[a:b], {"kind": "product", **product_metadata(text[a:b])}
        if len(block) <= 2500:
            doc_chunks.append(make_chunk(doc, len(doc_chunks), a, block, meta))
        else:                                       # rare long block: split, keep the name on each part
            for part in split_text(doc, block, a, len(doc_chunks), meta):
                part["text"] = part["text"].replace("\n", f"\n{meta['name']}\n", 1)
                doc_chunks.append(part)
    chunks += doc_chunks

table = pd.DataFrame(chunks)
print(len(chunks), "chunks:", table.kind.value_counts().to_dict())
print("products with SKU:", table.sku.notna().sum(), "| with price:", table.price.notna().sum())
print("\n--- example product chunk ---\n" + table[table.kind == "product"].iloc[100].text)
table[table.kind == "product"][["name", "sku", "price", "availability", "title"]].sample(5, random_state=1)

3508 chunks: {'product': 3291, 'policy': 144, 'catalog intro': 73}
products with SKU: 3282 | with price: 3287

--- example product chunk ---
[Product Catalog Audio and Headphones | product_catalog | page 3 | last reviewed August 2026]
Page 2 of 15 — True Wireless and Sports Earbuds
4. Sonetra TrailBeat Sport
SKU: IS-AUD-TWS-0004
Category: Audio & Headphones / Sports earbuds
Short description: Secure-fit wireless earbuds for running and gym use, supplied with ear wings and multiple
silicone tip sizes.
Price: €74.50
Key specifications:
•  Bluetooth 5.3
•  9.2 mm dynamic drivers
•  IPX7 water resistance
•  Ear-wing stabilisers in three sizes
•  Up to 9 hours playback; 27 hours with case
•  Ambient sound mode for outdoor awareness
•  USB-C fast charging
•  Case includes integrated lanyard loop
Availability: In stock
Warehouse location: ES-ZAZ, sports audio zone S-03


,name,sku,price,availability,title
458,"EcoLoop Recycled Plastic Bin, 6 Litre",BA-EL-1604,12.95,In stock,Product Catalog Bathroom Accessories
2438,Thomann TR 200 Bb Trombone,MI-BRS-THO-TR200,329.00,In stock — 8 units,Product Catalog Musical Instruments
3236,FloraDaily Lactobacillus 60 Capsules,PRO-FD-LAC60,17.60,In stock,Product Catalog Vitamins and Supplements
2120,Product status: Active,JW-ACC-PMS-GR-054,24.95,In stock — 31 units,Product Catalog Jewelry and Watches
320,"ClearSpan Dual-Weave Microfibre Towel Set, 6 Pack",AUT-DET-MFT-6PK,17.50,In stock — 596 packs,Product Catalog Automotive


<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 4 — Pinecone index + upload (dense + sparse + product metadata)

Same as v2 (dotproduct, two vectors per record), but the **metadata now includes `sku`, `price`, `kind`** → we can filter on them.

</div>

In [5]:
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
if not pc.has_index(INDEX_NAME):
    pc.create_index(name=INDEX_NAME, dimension=EMBED_DIM, metric="dotproduct",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    print("index created")
index = pc.Index(host=pc.describe_index(INDEX_NAME).host)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

def sparse_embed(texts, input_type):
    out = []
    for i in range(0, len(texts), 96):
        for attempt in range(5):
            try:
                result = pc.inference.embed(model=SPARSE_MODEL, inputs=texts[i:i + 96],
                                            parameters={"input_type": input_type, "truncate": "END"})
                break
            except Exception as error:
                if "429" not in str(error) or attempt == 4:
                    raise
                time.sleep(20)
        out += [{"indices": e.sparse_indices, "values": e.sparse_values} for e in result.data]
    return out

REBUILD = False
already = index.describe_index_stats().total_vector_count
if REBUILD or already != len(chunks):
    if already:
        index.delete(delete_all=True)
    texts = [c["text"] for c in chunks]
    t0 = time.time()
    dense = embeddings.embed_documents(texts)
    sparse = sparse_embed(texts, "passage")
    print(f"embedded {len(texts)} chunks (dense + sparse) in {time.time() - t0:.0f} s")
    index.upsert(vectors=[{"id": c["id"], "values": d, "sparse_values": s, "metadata": c}
                          for c, d, s in zip(chunks, dense, sparse)],
                 batch_size=100, show_progress=False)
    while index.describe_index_stats().total_vector_count < len(chunks):
        time.sleep(2)
else:
    print("Pinecone already has all", already, "chunks — nothing to upload")
print(index.describe_index_stats())

Pinecone already has all 3508 chunks — nothing to upload
DescribeIndexStatsResponse(dimension=1536, total_vector_count=3508, metric='dotproduct', namespaces=1)


<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 5 — The router

A small model reads the question and returns a **structured decision** (no free text):
- `target`: **policy**, **product** or **both** → which part of the index to search
- `sku`: an exact product code, if the question has one → direct lookup
- `max_price`: "under 300 euros" → `price ≤ 300` filter
- `wants_old_catalog`: only when someone asks about 2021 / old / discontinued items

The decision becomes a **Pinecone metadata filter**.

</div>

In [6]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class Route(BaseModel):
    target: Literal["policy", "product", "both"] = Field(
        description="policy = internal rules/procedures (HR, IT, finance, legal, sales, support, deliveries); "
                    "product = items we sell (prices, specs, stock); both = needs rules and product info")
    sku: Optional[str] = Field(None, description="only an exact product code written in the question, letters/digits joined by dashes like PRJ-PCB-PB1080; never a product name; else null")
    max_price: Optional[float] = Field(None, description="upper price limit in EUR if the question gives one, else null")
    wants_old_catalog: bool = Field(False, description="true only if the user asks about old, 2021 or past catalog data")

router_llm = ChatPromptTemplate.from_template(
    "Route this IronStore employee question.\n\nQuestion: {question}"
) | ChatOpenAI(model=ROUTER_MODEL, temperature=0).with_structured_output(Route)

CODE_PATTERN = re.compile(r"\b[A-Z0-9]{2,}(?:-[A-Z0-9]+)+\b")

def route_question(question):
    route = router_llm.invoke({"question": question})
    if route.sku and not CODE_PATTERN.fullmatch(route.sku.strip().upper()):   # safety net: names are not SKUs
        route.sku = None
    return route

def make_filter(route, outdated=False):
    if route.target == "policy":
        return {"doc_type": {"$eq": "policy"}}
    if route.target == "product":
        f = {"doc_type": {"$eq": "catalog"}, "outdated": {"$eq": outdated}}
        if route.max_price:
            f["price"] = {"$lte": route.max_price}
        return f
    return None if outdated else {"outdated": {"$eq": False}}

for q in ["How many days of annual leave do I get?", "Which drones do we sell under 300 euros?",
          "What is SKU PRJ-PCB-PB1080?", "Can I expense a monitor for my home office?",
          "What did the USB-C docking station cost in the 2021 catalog?"]:
    r = route_question(q)
    print(f"{q}\n   → {r.model_dump(exclude_defaults=True)}  filter: {make_filter(r)}\n")

How many days of annual leave do I get?
   → {'target': 'policy'}  filter: {'doc_type': {'$eq': 'policy'}}

Which drones do we sell under 300 euros?
   → {'target': 'product', 'max_price': 300.0}  filter: {'doc_type': {'$eq': 'catalog'}, 'outdated': {'$eq': False}, 'price': {'$lte': 300.0}}

What is SKU PRJ-PCB-PB1080?
   → {'target': 'product', 'sku': 'PRJ-PCB-PB1080'}  filter: {'doc_type': {'$eq': 'catalog'}, 'outdated': {'$eq': False}}

Can I expense a monitor for my home office?
   → {'target': 'policy'}  filter: {'doc_type': {'$eq': 'policy'}}

What did the USB-C docking station cost in the 2021 catalog?
   → {'target': 'both', 'wants_old_catalog': True}  filter: {'outdated': {'$eq': False}}



<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 6 — The reranker

Vector search compares **two separate vectors**. A reranker (*cross-encoder*) reads **question + chunk together** and scores how well the chunk *answers* the question (0 – 1). Slower, so it only re-sorts the ~10–20 deduplicated candidates.

⚠️ Learned while testing: the scores are good for **ordering**, but not as an absolute yes/no (a correct policy chunk for an indirect question scored only 0.04). So we keep the **top 6**, and the answer model still decides *not found*.

Example: three candidates for one question:

</div>

In [7]:
def rerank(question, candidates, top_n=TOP_N):
    if not candidates:
        return []
    result = pc.inference.rerank(model=RERANK_MODEL, query=question,
                                 documents=[c["text"] for c in candidates],
                                 top_n=min(top_n, len(candidates)), return_documents=False)
    return [{**candidates[r.index], "rerank": round(r.score, 3)} for r in result.data]

demo = [{"text": "Hybrid employees attend their designated office an average of two days per week."},
        {"text": "Remote work is a working arrangement, not an employee entitlement."},
        {"text": "The Nordview 55-inch TV costs €649 and is in stock."}]
pd.DataFrame(rerank("How many days do I have to be in the office?", demo))

,text,rerank
0,Hybrid employees attend their designated offic...,0.699
1,"Remote work is a working arrangement, not an e...",0.000
2,The Nordview 55-inch TV costs €649 and is in s...,0.000


<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 7 — Retrieval pipeline

1. 🧭 **router** and ✍️ **3 rewrites** run **at the same time** (two small LLM calls in parallel)
2. ⚡ **SKU fast path**: exact `sku` filter → no search, no rerank needed
3. 🔎 **hybrid search** for 4 question versions, **with the router's filter** (α from v2)
4. 🧹 **dedup** by id and text (v1)
5. 🏅 **rerank** → best 5
6. 🕰️ product question and no strong match → **also search the old 2021 catalog** and rerank both together (old chunks stay flagged outdated)

🌍 The rewrites are always in **English**, so a German question still matches the English documents.

</div>

In [8]:
from concurrent.futures import ThreadPoolExecutor

class Rewrites(BaseModel):
    questions: list[str] = Field(description="3 alternative versions of the question")

rewriter = ChatPromptTemplate.from_template(
    "You help search IronStore's internal documents (HR, IT, finance, legal, sales, customer support, "
    "deliveries, product catalogs).\nWrite 3 different versions of the employee's question IN ENGLISH (translate if needed; the documents are English). Use other words "
    "and synonyms an official policy or catalog would use. Keep product names, numbers and SKUs unchanged.\n\n"
    "Question: {question}"
) | ChatOpenAI(model=REWRITE_MODEL, temperature=0.3).with_structured_output(Rewrites)

def choose_alpha(question):
    return 0.3 if CODE_PATTERN.search(question) else 0.7

def search(dense_vec, sparse_vec, alpha, k, flt):
    alpha = min(max(alpha, 0.01), 1.0)
    result = index.query(
        vector=[v * alpha for v in dense_vec],
        sparse_vector={"indices": sparse_vec["indices"], "values": [v * (1 - alpha) for v in sparse_vec["values"]]},
        top_k=k, include_metadata=True, filter=flt,
    )
    return [{"id": m.id, "score": round(m.score, 3), **m.metadata} for m in result.matches]

def dedup(hits):
    best = {}
    for hit in hits:
        if hit["id"] not in best:
            best[hit["id"]] = {**hit, "hits": 0}
        best[hit["id"]]["hits"] += 1
        best[hit["id"]]["score"] = max(best[hit["id"]]["score"], hit["score"])
    unique, seen = [], set()
    for hit in sorted(best.values(), key=lambda h: (-h["hits"], -h["score"])):
        body = " ".join(hit["text"].split("\n", 1)[-1].split()).lower()
        if body not in seen:
            seen.add(body)
            unique.append(hit)
    return unique

def retrieve(question, k=8, verbose=False):
    log = {}
    with ThreadPoolExecutor(max_workers=2) as pool:                      # router ∥ rewrites
        route_job = pool.submit(route_question, question)
        rewrite_job = pool.submit(rewriter.invoke, {"question": question})
        route = route_job.result()
        queries = [question] + rewrite_job.result().questions[:3]
    log["route"] = route.model_dump(exclude_defaults=True)
    dense_vecs = embeddings.embed_documents(queries)

    if route.sku:                                                          # ⚡ SKU fast path
        hits = index.query(vector=dense_vecs[0], top_k=3, include_metadata=True,
                           filter={"sku": {"$eq": route.sku.upper()}}).matches
        if hits:
            log["path"] = "SKU lookup"
            kept = [{"id": h.id, "score": 1.0, "rerank": 1.0, **h.metadata} for h in hits]
            return kept, log

    sparse_vecs = sparse_embed(queries, "query")
    alpha = choose_alpha(question)

    def run(outdated):
        flt = make_filter(route, outdated)
        with ThreadPoolExecutor(max_workers=len(queries)) as pool:
            results = pool.map(lambda pair: search(*pair, alpha, k, flt), zip(dense_vecs, sparse_vecs))
        raw = [hit for hits in results for hit in hits]
        return raw

    raw = run(outdated=False)
    log["path"] = f"hybrid α={alpha}"
    kept = rerank(question, dedup(raw), TOP_N)
    best = kept[0]["rerank"] if kept else 0
    if route.target == "product" and (best < FALLBACK_RERANK or route.wants_old_catalog):   # 🕰️ fallback
        raw += run(outdated=True)
        kept = rerank(question, dedup(raw), TOP_N)
        log["path"] += " + old 2021 catalog"
    unique = dedup(raw)
    log.update(found=len(raw), after_dedup=len(unique), after_rerank=len(kept))

    if verbose:
        for i, q in enumerate(queries):
            print(f"Q{i}: {q}")
        print("\n", log, "\n")
        display(pd.DataFrame(kept)[["rerank", "score", "hits", "title", "page", "id"]])
    return kept, log

kept, log = retrieve("Which drones do we sell under 300 euros?", verbose=True)

Q0: Which drones do we sell under 300 euros?
Q1: What are the models of drones available for purchase that are priced below 300 euros?
Q2: Can you provide a list of drone products that are offered at a price point under 300 euros?
Q3: Which drone units do we have in our inventory that are listed for less than 300 euros?

 {'route': {'target': 'product', 'max_price': 300.0}, 'path': 'hybrid α=0.7 + old 2021 catalog', 'found': 64, 'after_dedup': 40, 'after_rerank': 6} 



,rerank,score,hits,title,page,id
0,0.018,4.009,4,Product Catalog Drones and RC Vehicles,19,IronStore_Product_Catalog_Drones_and_RC_Vehicl...
1,0.005,2.220,2,Product Catalog Drones and RC Vehicles,7,IronStore_Product_Catalog_Drones_and_RC_Vehicl...
2,0.005,3.318,4,Product Catalog Drones and RC Vehicles,4,IronStore_Product_Catalog_Drones_and_RC_Vehicl...
3,0.004,1.769,1,Product Catalog Drones and RC Vehicles,9,IronStore_Product_Catalog_Drones_and_RC_Vehicl...
4,0.004,1.777,1,Product Catalog Drones and RC Vehicles,15,IronStore_Product_Catalog_Drones_and_RC_Vehicl...
5,0.004,2.454,2,Product Catalog Drones and RC Vehicles,13,IronStore_Product_Catalog_Drones_and_RC_Vehicl...


<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 8 — Answer

Same prompt as v2, but the model now gets only the **6 best reranked chunks** (v2: 8 by vector score). The footer shows the router decision, the search path and the best rerank score.

</div>

In [9]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the IronStore internal assistant for employees.\n"
     "Answer ONLY with information from the sources below. Cite them inline like [S1] or [S2][S3].\n"
     "If the sources do not contain the answer, say: \"I couldn't find this in the internal documents\" "
     "and suggest which department could help. Never invent numbers, names or rules.\n"
     "If sources disagree, prefer the most recently reviewed one and mention the difference.\n"
     "Sources marked OUTDATED come from the 2021 catalog: say clearly that this information is outdated.\n"
     "Be concise; use bullet points for procedures and lists. Answer in the language of the question.\n\n"
     "Sources:\n{context}"),
    ("human", "{question}"),
])
answer_chain = answer_prompt | ChatOpenAI(model=ANSWER_MODEL, temperature=0)

def ask(question, verbose=False):
    t0 = time.time()
    kept, log = retrieve(question, verbose=verbose)
    t1 = time.time()
    context = "\n\n".join(f"[S{i}] {c['text']}" for i, c in enumerate(kept, 1))
    answer = answer_chain.invoke({"context": context, "question": question}).content
    t2 = time.time()

    cited = sorted({int(n) for n in re.findall(r"\[S(\d+)\]", answer) if 0 < int(n) <= len(kept)})
    lines = [f"- **[S{n}]** {kept[n-1]['title']}, page {kept[n-1]['page']}"
             f" — [open PDF]({Path(kept[n-1]['path']).as_uri()}#page={kept[n-1]['page']})"
             + (" ⚠️ outdated" if kept[n-1]["outdated"] else "") for n in cited]
    if any(kept[n-1]["outdated"] for n in cited):                       # 🆕 v3: warning added by code, not left to the LLM
        answer = "⚠️ **Based on the outdated 2021 catalog — please check the current price and availability.**\n\n" + answer
    display(Markdown(f"**Q: {question}**\n\n{answer}\n\n**Sources**\n" + ("\n".join(lines) or "- none")
                     + f"\n\n<sub>🧭 {log['route']} · {log['path']} · best rerank "
                       f"{max([c['rerank'] for c in kept], default=0):.2f} · retrieval {t1 - t0:.1f} s · answer {t2 - t1:.1f} s</sub>"))
    return {"answer": answer, "sources": [kept[n-1] for n in cited], "log": log}

<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 9 — Tests: one per new feature



</div>

**⚡ SKU fast path** (exact metadata lookup):

In [10]:
_ = ask("how many days in a week work from home can be taken?")

**Q: how many days in a week work from home can be taken?**

- For hybrid employees, most are expected to attend their designated office an average of two days per week, implying they can work remotely the other days of the week [S1].
- The exact number of remote days may vary by team, especially during peak trading periods, operational changes, or major projects, where higher office attendance may be required [S1].
- The working model (hybrid or remote-first) is confirmed during recruitment or internal transfer and recorded in Workday, with at least annual review [S1].
- Remote-first employees normally work remotely but must attend team events, planning sessions, training, or other business needs in person [S1].

Therefore, typically, hybrid employees work from home about three days per week, but this can vary by team and business needs. Remote-first employees work mostly remotely with some required office attendance [S1].

**Sources**
- **[S1]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)

<sub>🧭 {'target': 'policy'} · hybrid α=0.7 · best rerank 0.04 · retrieval 3.1 s · answer 1.9 s</sub>

**💶 Price filter** from the router (`price ≤ 300`, catalogs only):

In [11]:
_ = ask("Which drones do we sell under 300 euros?")

**Q: Which drones do we sell under 300 euros?**

We sell the following drones under €300:

- Vektor Talon Micro (SKU: FPV-42019) — €189.00  
  Indoor FPV drone with video monitor, brushless motors, 5–7 min flight time, and adjustable camera angle [S2].

- Aerialis PocketFly P1 (SKU: DRN-41031) — €79.90  
  Mini drone for beginners, 720p camera, 9 min flight time, Wi-Fi live view, includes two batteries [S3].

- Aerialis SkyScout S2 (discontinued, no replenishment) — €249.00  
  Entry-level GPS camera drone with 12 MP camera, 24 min flight time, two-axis gimbal [S1].

- TrackForge TerraCrawler 1:10 (SKU: RCV-43018) — €249.00  
  RC crawler vehicle, not a drone but under €300 [S4].

- HelixAir H120 (SKU: RCH-46008) — €139.00  
  Indoor coaxial helicopter, no camera, 8 min flight time [S5].

- AeroCraft Scout Cub (SKU: RCA-45003) — €229.00  
  RC trainer aircraft, fixed-wing, 8–12 min flight time [S6].

Only the first three are actual drones; the others are RC vehicles or helicopters. The Aerialis SkyScout S2 is discontinued and no longer replenished [S1].

**Sources**
- **[S1]** Product Catalog Drones and RC Vehicles, page 19 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=19)
- **[S2]** Product Catalog Drones and RC Vehicles, page 7 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=7)
- **[S3]** Product Catalog Drones and RC Vehicles, page 4 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=4)
- **[S4]** Product Catalog Drones and RC Vehicles, page 9 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=9)
- **[S5]** Product Catalog Drones and RC Vehicles, page 15 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=15)
- **[S6]** Product Catalog Drones and RC Vehicles, page 13 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Drones_and_RC_Vehicles.pdf#page=13)

<sub>🧭 {'target': 'product', 'max_price': 300.0} · hybrid α=0.7 + old 2021 catalog · best rerank 0.02 · retrieval 4.1 s · answer 3.0 s</sub>

**📄 Policy route** (catalogs are not searched at all):

In [ ]:
_ = ask("Can I work from home, and how many days do I need to be in the office?")

**🔀 Both** (needs a policy *and* possibly product info):

In [ ]:
_ = ask("Can I expense a monitor for my home office?")

**🕰️ Outdated fallback** (only in the 2021 catalog):

In [ ]:
_ = ask("What is the price of the IronStore 24 inch Full HD office monitor?")

**🚫 Not in the documents:**

In [ ]:
_ = ask("What is the CEO's annual salary?")

**🌍 German question:**

In [ ]:
_ = ask("Wie viele Tage Homeoffice sind erlaubt?")

<div style="background:#f3e5f5;border-left:6px solid #7b1fa2;padding:10px 14px;border-radius:4px">

### 🆕 V3 — Step 10 — Findings

**✅ Better than v2**
- 🟢 **Product chunks:** 3,291 products → one chunk each, 3,282 with `sku` and 3,287 with `price` in metadata. No product is cut in half any more.
- 🟢 **SKU fast path:** exact metadata lookup, no search or rerank needed.
- 🟢 **Price filter:** "drones under 300 euros" → the router sets `price ≤ 300`, and the answer lists real drones with prices and stock (v1/v2 can't filter on price).
- 🟢 **Policy routing:** policy questions never see the 3,000+ product chunks → less noise.
- 🟢 **Old catalog = fallback only:** the 2021 monitor is still found, and the answer now **always** starts with an outdated warning (added by code, so the LLM can't forget it).
- 🟢 **German works:** the rewrites are in English, so "Wie viele Tage Homeoffice" finds the Remote Work Policy and the answer comes back in German.
- 🟢 Speed: ~2–2.5 s retrieval (router and rewrites run in parallel) + ~1–2 s answer.

**🧪 Fixed while testing (first run)**
- 🔴 The router put product **names** into `sku` → fixed with a pattern check (`route_question`).
- 🔴 A reranker cut-off (0.05) blocked correct answers: reranker scores are **relative**, not absolute (a correct but indirect policy chunk scored 0.04) → now used for **ordering** only (top 6).
- 🔴 German questions failed (the keyword model is English) → rewrites are forced to English.

**⚠️ Still weak / ideas for v4**
- 🟡 List questions ("all drones under 300 €") only get the top 6 products. A **metadata-only listing** (filter + no vector) would return all of them.
- 🟡 The router adds one LLM call (it runs in parallel, so little extra time).
- 🟡 Next steps from the brief: **chat memory** for follow-ups → **FastAPI** endpoint → small **frontend** → **deploy**.

</div>